In [5]:
# Tiny shakespeare dataset: https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
# RNN effectiveness: https://karpathy.github.io/2015/05/21/rnn-effectiveness/

# tokenize just means convert the input space (vocabulary, etc) into a sequence of integers.

!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
!wget  https://cs.stanford.edu/people/karpathy/char-rnn/linux.txt

--2026-06-07 18:14:39--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  4.21MB/s    in 0.3s    

2026-06-07 18:14:39 (4.21 MB/s) - ‘input.txt.1’ saved [1115394/1115394]

--2026-06-07 18:14:39--  https://cs.stanford.edu/people/karpathy/char-rnn/linux.txt
Resolving cs.stanford.edu (cs.stanford.edu)... 171.64.64.64
Connecting to cs.stanford.edu (cs.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 999589 (976K) [text/plain]
Saving to: ‘linux.txt’

linux.txt           100%[===================>] 976.16K  4.09MB/s   

In [25]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
import numpy as np
%matplotlib inline

In [2]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

#with open('linux.txt', 'r', encoding='utf-8') as f:
#    linux_code = f.read()

In [8]:


chars = sorted(list(set(text)))
''.join(chars)
vocab_size = len(chars)

In [ ]:
# tokenizers: we can have a very long sequence of integers (as token output) with a very small vocabulary or
# a very large vocabulary with a very small sequence of integers as the encoder output.

In [ ]:

# byte-pair encoding: https://en.wikipedia.org/wiki/Byte-pair_encoding

In [12]:
import tiktoken
encoder = tiktoken.get_encoding("gpt2")
encoder.encode("hi there")

[5303, 612]

In [27]:
# hyper params

# this is the context length !
block_size = 8

# batch size
batch_size = 32


In [15]:
itos = {i:ch for i, ch in enumerate(chars)}
stoi = {ch:i for i, ch in enumerate(chars)}
# for i, s in enumerate(chars):
#     itos[i] = s
#     stoi[s] = i


encoder = lambda s: [stoi[x] for x in s]
decoder = lambda x: ''.join([itos[xx] for xx in x]) 

In [17]:
x1 = encoder("hi there")
print(x1)
print(decoder(x1))

[46, 47, 1, 58, 46, 43, 56, 43]
hi there


In [34]:
# training and val split
data_len = len(text)
split_boundary = int(0.9 * data_len)
encoded_data = torch.tensor(encoder(text))

train_data = encoded_data[:split_boundary]
val_data = encoded_data[split_boundary:]

print(f"number of chars = {data_len}, split boundary = {split_boundary}, length of training data = {len(train_data)}, length of val = {len(val_data)}")


number of chars = 1115394, split boundary = 1003854, length of training data = 1003854, length of val = 111540


tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])

In [41]:
torch.manual_seed(1337)
batch_size = 4

def get_sample_batch(split):
    data = train_data if split == 'train' else val_data

    ix = torch.randint(len(data) - block_size, (batch_size, ))
    xx = [data[i: i + block_size] for i in ix]
    yy = [data[i+1: i + block_size +1] for i in ix]
    return torch.stack(xx), torch.stack(yy)


xx, yy = get_sample_batch('train')

for x, y in zip(xx, yy):
    for j in range(len(x)):
        print(f"when input is {x[:j+1]}    ----->    output is {y[j]}")

#print(xx)
#print(yy)

when input is tensor([24])    ----->    output is 43
when input is tensor([24, 43])    ----->    output is 58
when input is tensor([24, 43, 58])    ----->    output is 5
when input is tensor([24, 43, 58,  5])    ----->    output is 57
when input is tensor([24, 43, 58,  5, 57])    ----->    output is 1
when input is tensor([24, 43, 58,  5, 57,  1])    ----->    output is 46
when input is tensor([24, 43, 58,  5, 57,  1, 46])    ----->    output is 43
when input is tensor([24, 43, 58,  5, 57,  1, 46, 43])    ----->    output is 39
when input is tensor([44])    ----->    output is 53
when input is tensor([44, 53])    ----->    output is 56
when input is tensor([44, 53, 56])    ----->    output is 1
when input is tensor([44, 53, 56,  1])    ----->    output is 58
when input is tensor([44, 53, 56,  1, 58])    ----->    output is 46
when input is tensor([44, 53, 56,  1, 58, 46])    ----->    output is 39
when input is tensor([44, 53, 56,  1, 58, 46, 39])    ----->    output is 58
when input i

### Notes:
- The byte-level BPE is another approach. It simply converts the text into UTF-8 first, and treat it as a stream of bytes.
- This guarantees that any text encoded in UTF-8 can be encoded by the BPE. 
- This has been used in BERT-like models like RoBERTa, BART, and DeBERTa, and GPT-like models like GPT-2.[14][15][16]

- Even though block size is 8 ( or > 1) we have to train the network with no or just a single char input so that it can generate when we just have nothing to start with.


In [51]:
vocab_size, xx.size(), yy.size()
print(yy)

tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [64]:
import torch.nn as nn

# C = number of channels == embedding dim or for hidden layers its the larger last dimension
# T = sequence length or block_size or context_length
# B = batch size or length

class BigramLM(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.embedding_table = nn.Embedding(vocab_size, vocab_size)

    def __call__(self, idx, targets=None):
        logits = self.embedding_table(idx)
        B, T, C = logits.shape
        lg_view = logits
        if targets == None:
            loss = None
        else:
            t = targets.view(B*T)
            lg_view = logits.view(B*T, C)
            loss = F.cross_entropy(lg_view, t)
            
        return lg_view, loss
        
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices of current context
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

m = BigramLM(vocab_size)
logits, loss = m(xx, yy)

logits.shape, loss

decoder(m.generate(torch.tensor([[0]]), 100)[0].tolist())

"\noxKL\n.p Dw-wAf'iom'pMFJRhrWYMV.DtKsjVNySEn!Vm'\nSSi'kT.IIEF voPH:3j-OlU..dCg$NJCgO;;:jiXcUXaz\nEM-qNku"